# SDK 문의 처리 실습

이 노트북은 19일차 오후 실습입니다. 먼저 아래 예제로 SDK가 반환한 도구 요청과 응답 형식을 확인하세요. 이어 가상 문의 세 건을 조회 도구에 연결합니다.

수업 커널의 openai와 httpx를 사용합니다. MockTransport가 모델 응답을 재현하므로 외부 API 요청은 발생하지 않습니다. 도구를 부른 횟수와 답변을 함께 확인하세요.

In [ ]:
import json
from openai import OpenAI
from httpx import Client, MockTransport, Response

records = {"T07": "검토 중"}
def lookup(ticket_id):
    if not ticket_id:
        return "접수 ID를 알려 주세요."
    return records.get(ticket_id, "조회 결과 없음")

def respond(request):
    body = json.loads(request.content)
    if any(item.get("role") == "tool" for item in body["messages"]):
        message = {"role": "assistant", "content": body["messages"][-1]["content"]}
        reason = "stop"
    else:
        question = body["messages"][-1]["content"]
        ticket = next((word for word in question.split() if word.startswith("T")), "")
        if not ticket:
            message = {"role":"assistant", "content":"접수 ID를 알려 주세요."}
            reason = "stop"
        else:
            call = {"id":"call1", "type":"function", "function":{
                "name":"lookup", "arguments":json.dumps({"ticket_id":ticket})}}
            message = {"role":"assistant", "content":None, "tool_calls":[call]}
            reason = "tool_calls"
    return Response(200, json={"id":"demo","object":"chat.completion","created":0,
        "model":"fixture","choices":[{"index":0,"message":message,"finish_reason":reason}]})

client = OpenAI(api_key="local-demo", http_client=Client(transport=MockTransport(respond)))
tools = [{"type":"function", "function":{"name":"lookup", "parameters":{
    "type":"object", "properties":{"ticket_id":{"type":"string"}}, "required":["ticket_id"]}}}]
def inquire(question):
    messages = [{"role":"user","content":question}]
    def send():
        return client.chat.completions.create(model="fixture", messages=messages, tools=tools).choices[0].message
    first = send()
    if not first.tool_calls:
        return first.content, 0
    call = first.tool_calls[0]
    messages.append(first.model_dump(exclude_none=True))
    result = lookup(**json.loads(call.function.arguments))
    messages.append({"role":"tool","tool_call_id":call.id,"content":result})
    return send().content, 1

print(inquire("접수 T07 상태"))

T07은 한 번 조회해 검토 중으로 돌아옵니다. 이제 접수 ID가 없는 문의와 없는 ID의 문의를 각각 실행하세요.

In [ ]:
# 두 문의를 만들어 inquire에 전달하세요.

<details><summary>풀이와 실행 결과</summary><pre><code>print(inquire("접수 상태"))
print(inquire("접수 T99 상태"))</code></pre><p>('접수 ID를 알려 주세요.', 0) · ('조회 결과 없음', 1)</p></details>